# Text Feature Extraction — Whisper / BERT / RoBERTa / XML-RoBERTa

| Stage | Model | Output |
|---|---|---|
| Speech-to-text | `WhisperWrapper` | transcript string |
| Token features | `BertWrapper` | `(B, T, 768)` token embeddings |
| Token features | `RobertaWrapper` | `(B, T, 1024)` token embeddings |
| Multilingual sentence features | `XmlRobertaWrapper` | `(B, 768)` sentence embeddings |

`WhisperWrapper` transcribes a 16 kHz mono waveform.  The text is then passed to
the three transformer wrappers.  All wrappers accept a string or list of strings.
Demo uses `multispeaker.wav` as the audio source.

In [1]:
from pathlib import Path

audio_path = Path("../tests/fixtures/audio/multispeaker.wav")
print(f"Audio file: {audio_path}")
print(f"Exists: {audio_path.exists()}")

Audio file: ../tests/fixtures/audio/multispeaker.wav
Exists: True


## 1. Speech-to-Text with Whisper

In [2]:
from exordium.audio.io import load_audio
from exordium.text.whisper import WhisperWrapper

waveform, sr = load_audio(audio_path, target_sample_rate=16000)

whisper = WhisperWrapper(device_id=-1)  # -1 → CPU
text = whisper(waveform)  # waveform: Tensor (T,) → str

print(f"Transcribed text: {text!r}")

/Users/fodorad/miniconda3/envs/exordium/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
objc[76570]: Class AVFFrameReceiver is implemented in both /Users/fodorad/miniconda3/envs/exordium/lib/python3.12/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x30fb583a8) and /opt/homebrew/Cellar/ffmpeg/8.0.1/lib/libavdevice.62.1.100.dylib (0x32b938328). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[76570]: Class AVFAudioReceiver is implemented in both /Users/fodorad/miniconda3/envs/exordium/lib/python3.12/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x30fb583f8) and /opt/homebrew/Cellar/ffmpeg/8.0.1/lib/libavdevice.62.1.100.dylib (0x32b938378). This may cause spurious casting failures and mysterious crashes. One of the duplica

Transcribed text: "Hey guys, what do you guys want to eat for lunch? How about Murphy's? They have really good burgers. Vegetarian, do you know if they have any good veggie burgers? You might have a black bean burger, but I'm not really sure what it's like. Okay, looking at Yelp, I can't really tell us any good. Do you want to try Tuk-Tuck? We can get Thai? Hey, I've never been there. I haven't had a lot of Thai food, honestly, so I'm a little unsure what I would order. Yeah, I heard good things about Thai food, too, and I really like Japanese and Korean food, but I really don't know what I would get at a Thai restaurant. Okay, you want to check out taste-based, see if it recommends anything for us?"


## 2. BERT Features

In [3]:
from exordium.text.bert import BertWrapper

bert = BertWrapper(device_id=-1)

# bert() always returns (B, T, 768) token-level tensors
bert_tokens = bert(text)
bert_pooled = bert(text)[:, 0]  # CLS token as sentence-level proxy → (B, 768)

print(f"BERT token-level: shape={tuple(bert_tokens.shape)}  (B x T_tokens x 768)")
print(f"BERT pooled:      shape={tuple(bert_pooled.shape)}  (B x 768)")

BERT token-level: shape=(1, 187, 768)  (B x T_tokens x 768)
BERT pooled:      shape=(1, 768)  (B x 768)


## 3. RoBERTa Features

In [4]:
from exordium.text.roberta import RobertaWrapper

roberta = RobertaWrapper(device_id=-1)

roberta_tokens = roberta(text)
roberta_pooled = roberta(text)[:, 0]  # CLS token as sentence-level proxy → (B, 1024)

print(f"RoBERTa token-level: shape={tuple(roberta_tokens.shape)}  (B x T_tokens x 1024)")
print(f"RoBERTa pooled:      shape={tuple(roberta_pooled.shape)}  (B x 1024)")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RoBERTa token-level: shape=(1, 177, 1024)  (B x T_tokens x 1024)
RoBERTa pooled:      shape=(1, 1024)  (B x 1024)


## 4. XML-RoBERTa Features (Multilingual)

In [5]:
from exordium.text.xml_roberta import XmlRobertaWrapper

xmlr = XmlRobertaWrapper(device_id=-1)

# Accepts a list of texts — returns a batch tensor
sentences = [
    text,  # English (from Whisper)
    "Ceci est un exemple en français.",
    "Dies ist ein Beispiel auf Deutsch.",
]

xmlr_pooled = xmlr(sentences)
print(f"XML-RoBERTa pooled: shape={tuple(xmlr_pooled.shape)}  (3 x 768)")
print("  Each row = one sentence embedding, language-agnostic 768-dim space")

XML-RoBERTa pooled: shape=(3, 768)  (3 x 768)
  Each row = one sentence embedding, language-agnostic 768-dim space


## 5. Single-sentence inference helper

In [6]:
# Use __call__ directly — inference() takes a dict, not a string
bert_vec = bert(text)[0, 0]  # (768,) — CLS token of first sentence
xmlr_vec = xmlr(text)[0]  # (768,) — XmlRoberta pools internally, squeeze batch dim

print(f"bert vector:  shape={tuple(bert_vec.shape)}")
print(f"xmlr vector:  shape={tuple(xmlr_vec.shape)}")

bert vector:  shape=(768,)
xmlr vector:  shape=(768,)
